# Static Accuracy Analysis (thesis-grade)

Loads the most recent `static_accuracy` run and reports ISO 9283-flavoured pose accuracy and repeatability for the Y-Z plane (Volcaniarm is 2-DOF planar; X is fixed bias).

**Key thesis-quotable metrics computed below:**
- **`RP_yz`** (Pose Repeatability, ISO 9283): `mean(d_to_centroid) + 3·std(d_to_centroid)`. **Frame-independent** — the URDF apriltag mount placeholder bias does NOT contaminate this. Headline number for arm repeatability.
- **`AP_yz`** (Pose Accuracy, ISO 9283): `|cluster_centroid − commanded|`. Carries the URDF tag-mount offset bias; treat as a bound, not a precise number, until the bias is independently measured. Reported alongside the offset components so the bias direction is visible.
- **Worst-case** distance to centroid alongside mean / std / 95% CI on every metric — agricultural targeting cares about the tail, not just the mean.
- **Application zones**: thresholds at `acceptable ≤ 10 mm`, `marginal 10-30 mm`, `failing > 30 mm` (tunable in `analysis/metrics.py`).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    accuracy_iso9283, align_fk_to_tag, latest_run, list_runs, load_run,
    repeatability_iso9283, summary, threshold_color, threshold_zone,
    WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
)

TEST_NAME = 'static_accuracy'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:           {config.get("run_id")}')
print(f'status:           {config.get("status")}  failure_reason: {config.get("failure_reason")}')
print(f'goal (y, z):      {config["goals"][0]}')
print(f'iterations:       {config["num_cycles"]}')
print(f'samples per visit:{config["samples_per_visit"]}')
print(f'tag rows:         {len(tag)}')
print(f'fk rows:          {len(fk)}')

target = tag[tag['phase'] == 'target'].copy()
home = tag[tag['phase'] == 'home'].copy()

run_id:           static_accuracy/2026-05-06/14-57-31
status:           completed  failure_reason: None
goal (y, z):      [0.3, 0.4]
iterations:       1
samples per visit:1
tag rows:         2
fk rows:          1


## Repeatability (RP_yz, ISO 9283)

Distance of each tag sample from the cluster centroid. Frame-independent — clean number, **no URDF bias**.

Need ≥ 2 samples for std; ideally 5+ samples × 3+ cycles to separate within-visit (motor + detection noise) from between-visit (arm repeatability).

In [2]:
rep = repeatability_iso9283(target, dims=('y', 'z'))
print(f'n samples:      {rep["n"]}')
print(f'centroid (y,z): ({rep["centroid"][0]:.4f}, {rep["centroid"][1]:.4f}) m')
print(f'per-dim std:    y={rep["per_dim_std"][0]*1000:.3f} mm, '
      f'z={rep["per_dim_std"][1]*1000:.3f} mm')
if rep['n'] >= 2:
    s = summary(rep['d_to_centroid'])
    print(f'\ndistance-to-centroid: {s.in_mm()}')
    print(f'\nRP_yz (ISO 9283):  {rep["RP_m"]*1000:.3f} mm   '
          f'(zone: {threshold_zone(rep["RP_m"]*1000)})')
    print(f'worst (max d_to_centroid): {rep["worst_m"]*1000:.3f} mm   '
          f'(zone: {threshold_zone(rep["worst_m"]*1000)})')
else:
    print('\nNeed ≥ 2 samples for repeatability. Re-run with samples_per_visit ≥ 5.')

n samples:      1
centroid (y,z): (-0.4465, -0.2968) m
per-dim std:    y=nan mm, z=nan mm

Need ≥ 2 samples for repeatability. Re-run with samples_per_visit ≥ 5.


## ~~AP_yz (cluster centroid vs commanded)~~ — disabled

Originally going to compute ISO 9283 Pose Accuracy by comparing tag centroid (in `apriltag_marker_base` frame) to commanded goal (in `volcaniarm_base_link` frame). That comparison **crosses frames with potentially different orientations** (apriltag_ros's image-derived axes vs the URDF body axes), so the AP_yz magnitude isn't meaningful until the URDF↔apriltag_ros rotation is empirically determined.

Use **`d_diff`** above (rotation-invariant distance) for accuracy claims today. AP_yz can come back once the diagnostic at the bottom of this notebook gives us the bridge transform.

In [3]:
# AP_yz cell intentionally disabled. See markdown above; use the
# d_diff cell above for the rotation-invariant accuracy proxy.
# To re-enable once the apriltag_ros / URDF rotation is known:
# 1. Determine T(apriltag_base_link -> apriltag_marker_base) via
#    tf2_echo (see diagnostic at the bottom of this notebook).
# 2. Apply that rotation in loader.py's tag_in_base_frame helper.
# 3. Compute AP on the rotated tag-in-base centroid vs the goal.
print('AP_yz disabled (frame mismatch not yet resolved). See markdown above.')

AP_yz disabled (frame mismatch not yet resolved). See markdown above.


## Scalar distance: |EE − base| from FK vs from tag (rotation-invariant)

Two ways to measure base-to-EE distance:
- **`d_fk`** = `|right_arm_tip_link − volcaniarm_base_link|` from joint angles.
- **`d_tag`** = `|apriltag_marker_ee − apriltag_marker_base|` from apriltag detection.

These differ by URDF translation mount offsets (apriltag bracket positions on base + EE), but **rotation conventions cancel** in scalar distance — so any unresolved orientation question between apriltag_ros's `apriltag_marker_base` and the URDF's `apriltag_base_link` does not contaminate this number.

- Mean `d_diff` per pose = the URDF translation-mount bias — a quotable single number.
- Std of `d_tag` across samples = repeatability of the distance measurement (clean).

In [4]:
merged = align_fk_to_tag(tag, fk)
print(merged[['cycle', 'sample_idx', 'd_tag', 'd_fk', 'd_diff']].to_string(index=False))
print()
for col, label in (('d_tag', 'd_tag '), ('d_fk', 'd_fk  '), ('d_diff', 'd_diff')):
    s = summary(merged[col])
    print(f'{label}: {s.in_mm()}')
print('\n  mean(d_diff) is the URDF translation-mount bias for this goal;')
print('  std(d_diff) is measurement noise + arm repeatability of the distance.')

 cycle  sample_idx    d_tag  d_fk   d_diff
     1           1 0.635648   0.5 0.135648

d_tag : n=1  mean=+635647.505 mm  std=0.000 mm  worst=635647.505 mm  95% CI ±0.000 mm  median=+635647.505 mm
d_fk  : n=1  mean=+500000.000 mm  std=0.000 mm  worst=500000.000 mm  95% CI ±0.000 mm  median=+500000.000 mm
d_diff: n=1  mean=+135647.505 mm  std=0.000 mm  worst=135647.505 mm  95% CI ±0.000 mm  median=+135647.505 mm

  mean(d_diff) is the URDF translation-mount bias for this goal;
  std(d_diff) is measurement noise + arm repeatability of the distance.


## Y-Z scatter — tag cluster + threshold zones (frame-consistent)

Tag samples + their centroid + the ISO 9283 repeatability ring + application thresholds, all in `apriltag_marker_base` frame. **Goal / FK markers are intentionally omitted** because they live in `volcaniarm_base_link` frame — plotting them on the same axes would visually conflate two different frames (which is what produced the misleading "big gap" earlier).

This plot focuses on what `RP_yz` actually measures: how tightly the samples cluster around their own centroid, regardless of where that centroid sits in any robot frame. Shape of the cluster matters; absolute position doesn't, for this metric.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

ax.scatter(target['y'], target['z'], marker='x', s=80, c='steelblue',
           label=f'tag samples (n={len(target)})', zorder=4)

if rep['n'] >= 2:
    cy, cz = rep['centroid']
    ax.scatter([cy], [cz], marker='o', s=140, facecolors='none',
               edgecolors='#2e9c4a', linewidths=2.5,
               label='tag centroid', zorder=5)
    theta = np.linspace(0, 2 * np.pi, 200)
    # ISO 9283 RP circle around centroid
    ax.plot(cy + rep['RP_m'] * np.cos(theta),
            cz + rep['RP_m'] * np.sin(theta),
            color=threshold_color(rep['RP_m'] * 1000),
            linewidth=2, label=f'RP_yz = {rep["RP_m"]*1000:.2f} mm')
    # Application threshold rings
    for r_mm, style, lbl in (
        (WEEDING_ACCEPTABLE_MM, ':', f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)'),
        (WEEDING_MARGINAL_MM, '--', f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)'),
    ):
        r = r_mm / 1000
        ax.plot(cy + r * np.cos(theta), cz + r * np.sin(theta),
                color='gray', linestyle=style, linewidth=1, label=lbl)
elif rep['n'] == 1:
    cy, cz = rep['centroid']
    ax.scatter([cy], [cz], marker='o', s=140, facecolors='none',
               edgecolors='#2e9c4a', linewidths=2.5,
               label='single sample (n=1, no RP_yz)', zorder=5)

ax.set_xlabel('tag y [m] (apriltag_marker_base frame)')
ax.set_ylabel('tag z [m] (apriltag_marker_base frame)')
ax.set_title(f'tag cluster + RP_yz  ({config["run_id"]})')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## CDF of distance-to-centroid (thesis-quotable: "X% within Y mm")

Cumulative distribution of `d_to_centroid`. Lets you state things like *"95% of samples land within 4.2 mm of the cluster centroid."* Adds the application-threshold lines for context.

In [ ]:
if rep['n'] >= 2:
    d_mm = np.sort(rep['d_to_centroid'].to_numpy() * 1000)
    cdf = np.arange(1, len(d_mm) + 1) / len(d_mm)
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.step(d_mm, cdf, where='post', color='steelblue', linewidth=2)
    ax.axvline(WEEDING_ACCEPTABLE_MM, color='gray', linestyle=':', alpha=0.6,
               label=f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)')
    ax.axvline(WEEDING_MARGINAL_MM, color='gray', linestyle='--', alpha=0.6,
               label=f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)')
    # Quote-friendly markers at 50/95 percentiles
    p50 = np.percentile(d_mm, 50)
    p95 = np.percentile(d_mm, 95)
    ax.axhline(0.5, color='lightgray', alpha=0.4)
    ax.axhline(0.95, color='lightgray', alpha=0.4)
    ax.axvline(p50, color='lightgray', alpha=0.4)
    ax.axvline(p95, color='lightgray', alpha=0.4)
    ax.text(p50, 0.05, f' p50={p50:.2f} mm', va='bottom', fontsize=8)
    ax.text(p95, 0.05, f' p95={p95:.2f} mm', va='bottom', fontsize=8)
    ax.set_xlabel('distance to centroid [mm]')
    ax.set_ylabel('cumulative fraction')
    ax.set_title('CDF of d_to_centroid')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)
    fig.tight_layout()
else:
    print('Need ≥ 2 samples for a CDF. Re-run with samples_per_visit ≥ 5.')

## Within-visit drift (settling sanity check)

Does the position drift across the samples within a single visit? Flat = the arm has settled and detection is stable. Trend = motor still settling, or thermal drift, or detection slowly homing in.

In [ ]:
if len(target) >= 2:
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    # Sample index proxies time -- the runner records one row per sample.
    for ax, col in zip(axes, ['y', 'z']):
        for cy, group in target.groupby('cycle'):
            ax.plot(group['sample_idx'], group[col] * 1000, marker='o',
                    label=f'cycle {cy}')
        ax.set_ylabel(f'{col} [mm]')
        ax.grid(True, alpha=0.3)
    axes[0].legend(loc='best', fontsize=8)
    axes[1].set_xlabel('sample index')
    fig.suptitle('within-visit drift', fontsize=10)
    fig.tight_layout()
else:
    print('Need ≥ 2 target samples for drift plot.')

## Thesis caveats summary

**Reportable as-is** (clean numbers):
- `RP_yz` (ISO 9283 Pose Repeatability) — frame-independent.
- `std(d_tag)` — repeatability of the scalar distance, rotation-invariant.
- Worst-case distance to centroid — agricultural targeting reliability tail.

**Reportable with translation-bias caveat**:
- `mean(d_diff) = mean(d_tag) − mean(d_fk)` — biased by URDF translation mount offsets (apriltag base + EE mounts, currently measured per commits 33e5fc1 / ec91605, expected mm-scale). Rotation conventions cancel in scalar distance.

**Not yet computed** (needs the URDF↔apriltag_ros rotation diagnostic below):
- AP_yz (ISO 9283 Pose Accuracy) — vector comparison between tag centroid and commanded goal needs the two frames to share an orientation, which we haven't verified.

**Re-run for thesis-quality stats**:
- `iterations ≥ 3` and `samples_per_visit ≥ 5` to compute within-vs-between-visit variance separately.

## Diagnostic for next session: empirical apriltag↔URDF rotation

When bringup is running and the camera sees the base tag, run:

```bash
ros2 run tf2_ros tf2_echo apriltag_base_link apriltag_marker_base
```

If translation is ~0 and rotation is identity (rpy ~0), the URDF orientation matches apriltag_ros — AP_yz can be re-enabled. Non-identity rotation tells us exactly the static transform needed to bridge the two; we'd then either fix the URDF apriltag joint orientation or apply the rotation in `loader.py`.